# Chapter 28: Real Time vs Batch Systems

<a href="../lite/lab/index.html?path=ch28_realtime_vs_batch.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def draw_cov_ellipse(ax, mean, cov, n_std=2, **kwargs):
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1,1], vecs[0,1]))
    w, h = 2 * n_std * np.sqrt(np.maximum(vals, 0))
    ax.add_patch(Ellipse(xy=mean, width=w, height=h, angle=angle, **kwargs))

A self driving car cannot wait until the end of the trip to figure out where it is.
It needs an answer NOW, every 10 milliseconds. But a surveying drone can fly the whole
mission, land, and spend 10 minutes optimizing a perfect map. These two use cases lead
to fundamentally different architectures.

**Real time** (online) systems process data as it arrives. **Batch** (offline) systems
collect everything first, then optimize globally.

## 28.1 Online Estimation

Online systems maintain a running estimate and update it with each new observation.
The Kalman filter is the classic example. Key properties:
- Constant time per update (for fixed state size)
- Cannot revisit past decisions
- Current estimate is always available

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
n_measurements = 50
true_x = 5.0
sigma_prior = 10.0
sigma_meas = 1.0
# ──────────────────────────────────────────────────────────────────────────────

np.random.seed(42)
measurements = np.random.normal(true_x, sigma_meas, n_measurements)

# Online: sequential Kalman updates
mu_online = [0.0]; sigma_online = [sigma_prior]
mu, sig = 0.0, sigma_prior
for z in measurements:
    K = sig**2 / (sig**2 + sigma_meas**2)
    mu = mu + K * (z - mu)
    sig = np.sqrt((1 - K) * sig**2)
    mu_online.append(mu); sigma_online.append(sig)

# Batch: solve all at once
weights = np.ones(n_measurements) / sigma_meas**2
w_prior = 1 / sigma_prior**2
mu_batch = (w_prior * 0 + np.sum(weights * measurements)) / (w_prior + np.sum(weights))
sig_batch = np.sqrt(1 / (w_prior + np.sum(weights)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(mu_online, 'steelblue', lw=2, label='online estimate')
ax.fill_between(range(len(mu_online)),
                np.array(mu_online) - 2*np.array(sigma_online),
                np.array(mu_online) + 2*np.array(sigma_online),
                alpha=0.2, color='steelblue')
ax.axhline(true_x, color='k', ls='--', label=f'true = {true_x}')
ax.axhline(mu_batch, color='tomato', ls=':', lw=2, label=f'batch = {mu_batch:.3f}')
ax.set_xlabel("Measurement number"); ax.set_ylabel("Estimate")
ax.set_title("Online vs Batch: same final answer", fontsize=13); ax.legend(fontsize=9)

ax = axes[1]
ax.plot(sigma_online, 'steelblue', lw=2, label='online σ')
ax.axhline(sig_batch, color='tomato', ls=':', lw=2, label=f'batch σ = {sig_batch:.3f}')
ax.set_xlabel("Measurement number"); ax.set_ylabel("Uncertainty (σ)")
ax.set_title("Uncertainty reduction", fontsize=13); ax.legend()

plt.tight_layout()
plt.show()

print(f"Online final: μ = {mu_online[-1]:.4f}, σ = {sigma_online[-1]:.4f}")
print(f"Batch:        μ = {mu_batch:.4f}, σ = {sig_batch:.4f}")
print(f"Difference:   {abs(mu_online[-1] - mu_batch):.6f}")

## 28.2 Offline Optimization

Batch systems have access to ALL data simultaneously. They can iterate, revisit decisions,
and find globally optimal solutions. Graph SLAM (Chapter 25) is a batch approach.

## 28.3 Tradeoffs

| Property | Online (EKF) | Batch (Graph SLAM) |
|----------|:---:|:---:|
| **Latency** | Low (immediate) | High (wait for all data) |
| **Accuracy** | Good | Best |
| **Memory** | Fixed | Grows with data |
| **Can fix past errors** | No | Yes |
| **Loop closure** | One shot | Iterative refinement |

**Key observations:**
- For the same data and Gaussian assumptions, online and batch give **identical results**.
- Batch systems can use iterative solvers (Gauss-Newton) that improve with each iteration.
- Modern systems often use a **hybrid**: EKF for real time tracking, batch optimization in the background.

---

## Exercises

### Exercise 28.1
Implement both online (Kalman filter) and batch (least squares) estimation for a 2D
position from 30 range measurements to 3 known beacons. Verify they give the same answer.

In [ ]:
# Your code here